# Clase 216 — Filtrado colaborativo user/item-based con scipy.sparse

Sintético small (replica MovieLens 100K en tamaño). Requiere: `pip install scipy scikit-learn pandas`.

In [ ]:
import numpy as np, pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

rng = np.random.default_rng(42)
n_users, n_items = 1000, 500
n_ratings = 30_000   # ~6% sparsity

users = rng.integers(0, n_users, n_ratings)
items = rng.integers(0, n_items, n_ratings)
# Ratings sesgados por user (algunos califican alto, otros bajo)
user_bias = rng.normal(0, 0.5, n_users)
ratings = np.clip(3 + user_bias[users] + rng.normal(0, 0.8, n_ratings), 1, 5).round()

df = pd.DataFrame({'u': users, 'i': items, 'r': ratings}).drop_duplicates(['u', 'i'])
print(f'ratings únicos: {len(df):,}')

R = csr_matrix((df.r, (df.u, df.i)), shape=(n_users, n_items))
print(f'matrix: {R.shape}, nnz={R.nnz:,}, sparsity={1 - R.nnz / np.prod(R.shape):.4f}')

## 1. Similitud user-user (coseno)

In [ ]:
sim_users = cosine_similarity(R, dense_output=False)
print('shape:', sim_users.shape, '| tipo:', type(sim_users).__name__)

# Top 5 más similares a user_id=42 (excluyéndolo a sí mismo)
target = 42
sims_arr = sim_users[target].toarray().ravel()
sims_arr[target] = -1   # excluir self
top5 = np.argsort(-sims_arr)[:5]
for u in top5:
    print(f'  user_{u}: similarity={sims_arr[u]:.4f}, items en común={(R[target].toarray() * R[u].toarray() > 0).sum()}')

## 2. User-based prediction

In [ ]:
def user_based_predict(R, sim_users, user_id, top_k=30):
    """Top-N items para user_id usando user-based kNN."""
    sims = sim_users[user_id].toarray().ravel()
    sims[user_id] = 0
    # Top-K vecinos
    top_k_idx = np.argpartition(-sims, top_k)[:top_k]
    sims_k = sims[top_k_idx]
    R_k = R[top_k_idx].toarray()   # (top_k, n_items)
    # Predicción: weighted avg de ratings de neighbors
    weights = np.where(R_k > 0, 1, 0)
    num = (sims_k[:, None] * R_k).sum(axis=0)
    den = (sims_k[:, None] * weights).sum(axis=0) + 1e-9
    return num / den

scores = user_based_predict(R, sim_users, target)
# Excluir items ya vistos
seen = R[target].toarray().ravel() > 0
scores[seen] = -1
top10 = np.argsort(-scores)[:10]
print('top-10 user-based:', list(zip(top10.tolist(), scores[top10].round(3).tolist())))

## 3. Item-based (más estable, escalable)

In [ ]:
sim_items = cosine_similarity(R.T, dense_output=False)
print('item-item shape:', sim_items.shape)

def item_based_predict(R, sim_items, user_id):
    """Score = vector user × matriz item-item."""
    user_vec = R[user_id]   # (1, n_items)
    scores = (user_vec @ sim_items).toarray().ravel()
    return scores

scores_ib = item_based_predict(R, sim_items, target)
scores_ib[seen] = -1
top10_ib = np.argsort(-scores_ib)[:10]
print('top-10 item-based:', list(zip(top10_ib.tolist(), scores_ib[top10_ib].round(3).tolist())))

overlap = set(top10.tolist()) & set(top10_ib.tolist())
print(f'\noverlap user vs item-based top-10: {len(overlap)}/10')

## 4. Mean centering (Pearson-like)

In [ ]:
# Calculamos user means sobre items rateados
user_means = np.array(R.sum(axis=1)).ravel() / np.maximum(np.array((R > 0).sum(axis=1)).ravel(), 1)

# Centramos: solo restamos donde hay rating (no en los 0)
R_dense = R.toarray()
R_centered = np.where(R_dense > 0, R_dense - user_means[:, None], 0)

sim_pearson = cosine_similarity(R_centered)
print(f'sim Pearson user_42 ↔ user_top5_coseno:')
for u in top5:
    print(f'  user_{u}: pearson={sim_pearson[target, u]:.4f} vs coseno={sims_arr[u]:.4f}')

## 5. Comparativa de tiempos

In [ ]:
import time

t0 = time.perf_counter()
for u in range(100): user_based_predict(R, sim_users, u)
t_ub = (time.perf_counter() - t0) * 10   # promedio per request × ms

t0 = time.perf_counter()
for u in range(100): item_based_predict(R, sim_items, u)
t_ib = (time.perf_counter() - t0) * 10

print(f'user-based predict: {t_ub:.2f} ms/request')
print(f'item-based predict: {t_ib:.2f} ms/request')
print('\n→ item-based gana en producción porque sim_items es estable y pre-calculable.')

## Ejercicio guiado

1. Descargá MovieLens 100K real. Reemplazá el dataset sintético. Mostrá top-10 con título de película (no ID).
2. Implementá un split leave-one-out por user. Evaluá recall@10 (Clase 220).
3. Comparar coseno vs Pearson sobre el dataset real. ¿Cuál da mejor recall@10?
4. Mitigá el "todo es popular" weighting por IDF: `item_idf = log(n_users / item_count)`. Multiplicá scores por IDF antes del top-10.
5. Bonus: usar `NearestNeighbors` con `metric='cosine'` para mantener solo top-K=50 similares por item (sparsifica la matriz item-item).

## Conclusiones

- Matriz usuario-item es sparse extremo (>99%); densificarla revienta memoria.
- Coseno es default; Pearson cuando hay sesgo de usuario; Jaccard para binario.
- Item-based gana en escala porque las similitudes item-item son estables.
- CF kNN funciona bien hasta ~100K items; arriba, factorización (Clase 217).

## ✅ Soluciones de los ejercicios

Sin descargar MovieLens: generamos una **matriz usuario×item sintética y dispersa** (gustos
por género latentes) y sobre ella corremos filtrado colaborativo *user-based* e *item-based*
con `scipy.sparse` + `sklearn`. Todo ejecutable, sin internet.

### Ejercicio 1 — Matriz dispersa `R` (`csr_matrix`) y sparsity

Construimos `R` de forma `(n_users, n_items)` como `csr_matrix` y medimos la sparsity
`1 - nnz / (n_users*n_items)`, típicamente >95% en datos reales.

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

rng = np.random.default_rng(0)
n_users, n_items, n_genres = 100, 50, 5

# gustos latentes: cada user y cada item tienen afinidad por géneros
U = rng.random((n_users, n_genres))
V = rng.random((n_items, n_genres))
affinity = U @ V.T                       # (users, items) preferencia latente

# cada user califica ~12 items (los de mayor afinidad + ruido) -> matriz dispersa
R_dense = np.zeros((n_users, n_items))
for u in range(n_users):
    k = rng.integers(8, 16)
    items = np.argsort(affinity[u] + rng.normal(0, 0.1, n_items))[-k:]
    R_dense[u, items] = rng.integers(1, 6, size=k)   # ratings 1..5

R = csr_matrix(R_dense)
sparsity = 1 - R.nnz / (n_users * n_items)
print(f"R shape={R.shape} nnz={R.nnz} sparsity={sparsity:.1%}")

assert R.shape == (n_users, n_items)
assert sparsity > 0.5, "la matriz de ratings es dispersa"
print("OK ejercicio 1 — matriz de ratings dispersa construida")

### Ejercicio 2 — Similitud entre usuarios

`cosine_similarity(R)` da la matriz `(n_users, n_users)`. Buscamos los 5 usuarios más
parecidos al `user_id=42` (excluyéndolo a él mismo).

In [ ]:
sim_users = cosine_similarity(R)         # (n_users, n_users)
uid = 42
sims = sim_users[uid].copy()
sims[uid] = -np.inf                       # no me comparo conmigo mismo
top5 = np.argsort(sims)[-5:][::-1]
print("usuarios más similares a 42:", top5.tolist())
print("similitudes:", np.round(sims[top5], 3).tolist())

assert sim_users.shape == (n_users, n_users)
assert uid not in top5
assert np.all(np.diff(sims[top5]) <= 1e-9), "ordenados de mayor a menor similitud"
print("OK ejercicio 2 — top-5 usuarios similares a user 42")

### Ejercicio 3 — Recomendación user-based top-10

Puntuamos los items no vistos por `user_id=42` combinando los ratings de sus vecinos
ponderados por similitud: `scores = sim_users[42] @ R`. Recomendamos los top-10 que aún no vio.

In [ ]:
scores_ub = sim_users[uid] @ R.toarray()   # suma ponderada por similitud
seen = R[uid].toarray().ravel() > 0
scores_ub[seen] = -np.inf                   # excluir lo ya visto
rec_ub = np.argsort(scores_ub)[-10:][::-1]
print("user-based top-10 para user 42:", rec_ub.tolist())

assert len(rec_ub) == 10
assert not seen[rec_ub].any(), "no recomendamos items ya vistos"
print("OK ejercicio 3 — top-10 user-based (vecinos ponderados por similitud)")

### Ejercicio 4 — Recomendación item-based top-10

Ahora la similitud es entre items: `sim_items = cosine_similarity(R.T)` → `(n_items, n_items)`.
El score para el user es `R[42] @ sim_items`: items parecidos a los que ya le gustaron.

In [ ]:
sim_items = cosine_similarity(R.T)          # (n_items, n_items)
scores_ib = R[uid].toarray().ravel() @ sim_items
scores_ib[seen] = -np.inf
rec_ib = np.argsort(scores_ib)[-10:][::-1]
print("item-based top-10 para user 42:", rec_ib.tolist())

overlap = len(set(rec_ib) & set(rec_ub))
print(f"solapamiento con user-based: {overlap}/10")

assert sim_items.shape == (n_items, n_items)
assert len(rec_ib) == 10 and not seen[rec_ib].any()
print("OK ejercicio 4 — top-10 item-based (items similares a los ya gustados)")

### Ejercicio 5 — Pearson vía mean centering

Restar la media de cada usuario antes del coseno equivale a la **correlación de Pearson**:
corrige el sesgo de usuarios "generosos" (todo 5) vs "duros" (todo 2). Comparamos las
recomendaciones contra el coseno vanilla.

In [ ]:
R_arr = R.toarray()
mask = R_arr > 0
user_means = np.where(mask.sum(1) > 0, R_arr.sum(1) / np.maximum(mask.sum(1), 1), 0)
R_centered = np.where(mask, R_arr - user_means.reshape(-1, 1), 0)   # centrar solo lo observado

sim_pearson = cosine_similarity(R_centered)
scores_p = sim_pearson[uid] @ R_centered
scores_p[seen] = -np.inf
rec_p = np.argsort(scores_p)[-10:][::-1]
print("pearson (mean-centered) top-10:", rec_p.tolist())

overlap_p = len(set(rec_p) & set(rec_ub))
print(f"solapamiento vanilla vs pearson: {overlap_p}/10 (difieren por corregir el sesgo por usuario)")
assert sim_pearson.shape == (n_users, n_users)
assert len(rec_p) == 10
print("OK ejercicio 5 — Pearson por mean-centering vs coseno vanilla")